# 06 - Analysis And Reporting

This notebook produces the paper-facing analysis while keeping lexical detection and model-panel semantics separate. It removes deterministic greedy seed duplicates, reports interval estimates, fits a prompt-clustered binomial model, and treats the two annotators plus adjudicator as a sensitivity analysis rather than human ground truth.

A GPU is not required. Run notebook `05` first so the latest calibrated semantic configurations are available on Hugging Face.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import wandb
from datasets import load_dataset

from ocn.annotation import add_semantic_outcomes
from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, save_dataframe, utc_timestamp
from ocn.metrics import (
    deduplicate_greedy_seed_reuse,
    detection_summary,
    grouped_ocn_rates_with_ci,
    top_patterns,
    weighted_group_bootstrap,
)

paths = make_colab_paths()
config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
_ = login_huggingface("HF_WRITE_ACCESS")
EXPERIMENT_ID = "main_gemma4_qwen35"
ANALYSIS_VERSION = "v2_robust"
ANALYSIS_RUN_ID = utc_timestamp()
MAIN_DETECTION_REPO = config.get(
    "hf_main_detection_repo",
    f"{config['hf_owner']}/ocn-empty-negations-detection-main-gemma4-qwen35",
)
MAIN_SEMANTIC_REPO = config.get(
    "hf_main_semantic_repo",
    f"{config['hf_owner']}/ocn-empty-negations-semantic-main-gemma4-qwen35",
)
analysis_config = {
    **config,
    "experiment_id": EXPERIMENT_ID,
    "analysis_run_id": ANALYSIS_RUN_ID,
    "analysis_version": ANALYSIS_VERSION,
    "detection_repo": MAIN_DETECTION_REPO,
    "semantic_repo": MAIN_SEMANTIC_REPO,
    "greedy_seed_deduplication": True,
    "regression_covariance": "prompt_clustered",
    "semantic_bootstrap_replicates": 2000,
}
run = login_wandb(
    project="ocn-empty-negations",
    name=f"analysis-{EXPERIMENT_ID}-{ANALYSIS_RUN_ID}",
    config=analysis_config,
)
sns.set_theme(style="whitegrid")

In [ ]:
raw_df = load_dataset(MAIN_DETECTION_REPO, split="train").to_pandas()
df = deduplicate_greedy_seed_reuse(raw_df)
semantics = load_dataset(
    MAIN_SEMANTIC_REPO, "adjudicated", split="train"
).to_pandas()
calibration = load_dataset(
    MAIN_SEMANTIC_REPO, "calibration", split="train"
).to_pandas()

duplicate_audit = pd.DataFrame([{
    "raw_rows": len(raw_df),
    "analysis_rows": len(df),
    "removed_greedy_seed_duplicates": len(raw_df) - len(df),
    "raw_greedy_rows": int(raw_df["decoding"].eq("greedy").sum()),
    "analysis_greedy_rows": int(df["decoding"].eq("greedy").sum()),
}])
summary = detection_summary(df)
model_rates = grouped_ocn_rates_with_ci(
    df,
    ["model_id", "model_stage", "model_family", "decoding"],
    cluster_column="prompt_id",
    n_boot=2000,
    seed=20260821,
)
variant_rates = grouped_ocn_rates_with_ci(
    df, ["variant"], cluster_column="prompt_id", n_boot=2000, seed=20260822
)
persona_rates = grouped_ocn_rates_with_ci(
    df, ["persona"], cluster_column="prompt_id", n_boot=2000, seed=20260823
)
category_rates = grouped_ocn_rates_with_ci(
    df, ["category"], cluster_column="prompt_id", n_boot=2000, seed=20260824
)
patterns = top_patterns(df, 12)

output_root = Path(config["drive_data_root"]) / "analysis" / EXPERIMENT_ID / ANALYSIS_VERSION
output_root.mkdir(parents=True, exist_ok=True)
save_dataframe(duplicate_audit, output_root / "deduplication_audit.csv")
save_dataframe(model_rates, output_root / "lexical_model_rates.csv")
save_dataframe(variant_rates, output_root / "lexical_variant_rates.csv")
save_dataframe(persona_rates, output_root / "lexical_persona_rates.csv")
save_dataframe(category_rates, output_root / "lexical_category_rates.csv")
save_dataframe(patterns, output_root / "lexical_patterns.csv")
display(duplicate_audit)
display(summary.to_frame("value"))

In [ ]:
panel_frames = []
panel_specs = [
    ("annotator_a", "a_taxonomy_label", "a_prompt_support"),
    ("annotator_b", "b_taxonomy_label", "b_prompt_support"),
    ("adjudicator", "taxonomy_label", "prompt_support"),
]
for panel_source, taxonomy_column, support_column in panel_specs:
    panel = add_semantic_outcomes(
        semantics,
        taxonomy_column=taxonomy_column,
        prompt_support_column=support_column,
    )
    panel["panel_source"] = panel_source
    panel_frames.append(panel)
semantic_sensitivity_rows = pd.concat(panel_frames, ignore_index=True, sort=False)

semantic_overall = weighted_group_bootstrap(
    semantic_sensitivity_rows,
    outcomes=["strict_misuse", "broad_misuse", "unsupported_contrast"],
    group_columns=["panel_source"],
    cluster_column="response_id",
    weight_column="sample_weight",
    n_boot=2000,
    seed=20260819,
)
semantic_by_model = weighted_group_bootstrap(
    semantic_sensitivity_rows,
    outcomes=["strict_misuse", "broad_misuse", "unsupported_contrast"],
    group_columns=["panel_source", "model_id", "model_stage", "decoding"],
    cluster_column="response_id",
    weight_column="sample_weight",
    n_boot=2000,
    seed=20260820,
)
calibration_summary = (
    calibration.groupby(["panel_member", "annotator_model_id"], dropna=False)
    .agg(
        calibration_cases=("example_id", "size"),
        calibration_correct=("calibration_correct", "sum"),
    )
    .reset_index()
)
calibration_summary["calibration_accuracy"] = (
    calibration_summary["calibration_correct"]
    / calibration_summary["calibration_cases"]
)
save_dataframe(semantic_overall, output_root / "semantic_panel_sensitivity_overall.csv")
save_dataframe(semantic_by_model, output_root / "semantic_panel_sensitivity_by_model.csv")
save_dataframe(calibration_summary, output_root / "semantic_panel_calibration.csv")
display(semantic_overall)
display(calibration_summary)

In [ ]:
regression_df = df.copy()
regression_df["has_ocn_int"] = regression_df["has_ocn"].astype(int)
formula = (
    "has_ocn_int ~ C(model_stage) * C(model_family) + C(decoding) + "
    "C(variant) + C(persona) + C(category) + length_target"
)
model = smf.glm(
    formula,
    data=regression_df,
    family=sm.families.Binomial(),
).fit(
    cov_type="cluster",
    cov_kwds={"groups": regression_df["prompt_id"]},
)
confidence = model.conf_int()
coefficient_table = pd.DataFrame({
    "term": model.params.index,
    "log_odds": model.params.values,
    "clustered_std_error": model.bse.values,
    "p_value": model.pvalues.values,
    "odds_ratio": np.exp(model.params.values),
    "odds_ratio_ci_low": np.exp(confidence[0].values),
    "odds_ratio_ci_high": np.exp(confidence[1].values),
})
report_path = output_root / "clustered_binomial_model_summary.txt"
report_path.write_text(model.summary().as_text(), encoding="utf-8")
save_dataframe(coefficient_table, output_root / "clustered_binomial_coefficients.csv")
print(model.summary())

In [ ]:
def interval_dotplot(frame, label_column, rate_column, low_column, high_column, ax, color):
    plot = frame.sort_values(rate_column).reset_index(drop=True)
    positions = np.arange(len(plot))
    ax.errorbar(
        plot[rate_column],
        positions,
        xerr=np.vstack([
            plot[rate_column] - plot[low_column],
            plot[high_column] - plot[rate_column],
        ]),
        fmt="o",
        color=color,
        ecolor=color,
        capsize=3,
    )
    ax.set_yticks(positions, plot[label_column])
    ax.set_xlim(0, 1)

fig, axes = plt.subplots(2, 3, figsize=(22, 13))
model_plot = model_rates.copy()
model_plot["label"] = (
    model_plot["model_id"].str.split("/").str[-1]
    + " | " + model_plot["decoding"].astype(str)
)
interval_dotplot(
    model_plot, "label", "ocn_rate", "ocn_rate_ci_low", "ocn_rate_ci_high",
    axes[0, 0], "#35618f",
)
axes[0, 0].set_title("Lexical OCN rate by model and decoding")

interval_dotplot(
    variant_rates, "variant", "ocn_rate", "ocn_rate_ci_low", "ocn_rate_ci_high",
    axes[0, 1], "#e17829",
)
axes[0, 1].set_title("Lexical OCN rate by prompt variant")

sns.barplot(data=patterns, y="pattern", x="count", ax=axes[0, 2], color="#6f8f5d")
axes[0, 2].set_title("Top lexical detector patterns")

semantics["taxonomy_label"].value_counts().sort_values().plot(
    kind="barh", ax=axes[1, 0], color="#bd4d4d"
)
axes[1, 0].set_title("Adjudicator taxonomy (model-assisted)")

sensitivity_plot = semantic_overall.copy()
interval_dotplot(
    sensitivity_plot,
    "panel_source",
    "strict_misuse_rate",
    "strict_misuse_ci_low",
    "strict_misuse_ci_high",
    axes[1, 1],
    "#71588f",
)
axes[1, 1].set_title("Strict misuse sensitivity by panel member")

calibration_plot = calibration_summary.copy()
calibration_plot["label"] = calibration_plot["panel_member"]
sns.barplot(
    data=calibration_plot,
    y="label",
    x="calibration_accuracy",
    ax=axes[1, 2],
    color="#2d8b84",
)
axes[1, 2].axvline(1 / 8, color="black", linestyle="--", linewidth=1)
axes[1, 2].set_xlim(0, 1)
axes[1, 2].set_title("Held-out calibration accuracy")
plt.tight_layout()

figure_root = Path(config["drive_figure_root"]) / "analysis" / EXPERIMENT_ID / ANALYSIS_VERSION
figure_root.mkdir(parents=True, exist_ok=True)
fig_path = figure_root / "06_analysis_dashboard.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")

report_markdown = f'''# OCN Main Analysis ({ANALYSIS_VERSION})

- Raw generation rows: {len(raw_df):,}
- Analysis rows after deterministic greedy deduplication: {len(df):,}
- Removed greedy seed duplicates: {len(raw_df) - len(df):,}
- Lexical OCN rate: {summary['ocn_rate']:.3f}
- Prompt-clustered binomial regression: `{formula}`
- Semantic estimates: response-cluster bootstrap with 2,000 replicates

Semantic results are model-panel sensitivity estimates. They are not human-gold prevalence estimates and remain provisional until the blinded audit is independently annotated and adjudicated.
'''
report_markdown = "\n".join(line.strip() for line in report_markdown.splitlines()).strip() + "\n"
report_md_path = output_root / "analysis_report.md"
report_md_path.write_text(report_markdown, encoding="utf-8")

wandb.log({
    **summary.to_dict(),
    "raw_rows": len(raw_df),
    "analysis_rows": len(df),
    "removed_greedy_seed_duplicates": len(raw_df) - len(df),
    "analysis_dashboard": wandb.Image(str(fig_path)),
    "model_rates": wandb.Table(dataframe=model_rates),
    "variant_rates": wandb.Table(dataframe=variant_rates),
    "persona_rates": wandb.Table(dataframe=persona_rates),
    "category_rates": wandb.Table(dataframe=category_rates),
    "semantic_panel_sensitivity": wandb.Table(dataframe=semantic_overall),
    "semantic_panel_sensitivity_by_model": wandb.Table(dataframe=semantic_by_model),
    "semantic_panel_calibration": wandb.Table(dataframe=calibration_summary),
    "clustered_binomial_coefficients": wandb.Table(dataframe=coefficient_table),
    "clustered_binomial_summary": model.summary().as_text(),
})
run.finish()
print("Saved analysis tables:", output_root)
print("Saved dashboard:", fig_path)
print("Saved report:", report_md_path)